In [ ]:
import numpy as np


class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iters=2000):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float).reshape(-1)

        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0

        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias

            # Mean Squared Error gradients
            dw = (2.0 / n_samples) * (X.T @ (y_pred - y))
            db = (2.0 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            loss = np.mean((y - y_pred) ** 2)
            self.loss_history.append(loss)

        return self

    def predict(self, X):
        X = np.array(X, dtype=float)
        return X @ self.weights + self.bias

    def score_r2(self, X, y):
        y = np.array(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)

        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)

        if ss_tot == 0:
            return 1.0
        return 1.0 - (ss_res / ss_tot)



# Demo with synthetic data
# y = 3*x1 + 2*x2 + 5 + noise
# -----------------------------
np.random.seed(42)

n = 200
X = np.random.randn(n, 2)
true_w = np.array([3.0, 2.0])
true_b = 5.0
noise = np.random.randn(n) * 0.3
y = X @ true_w + true_b + noise

model = LinearRegressionScratch(learning_rate=0.05, n_iters=3000)
model.fit(X, y)

print("Learned weights:", model.weights)
print("Learned bias:", model.bias)
print("R^2 score:", model.score_r2(X, y))
print("Final loss:", model.loss_history[-1])

# Sample predictions
X_new = np.array([[1.0, 2.0], [-1.0, 0.5], [0.0, 0.0]])
print("Predictions:", model.predict(X_new))

Learned weights: [3.0198434  2.01405076]
Learned bias: 4.973633100954647
R^2 score: 0.9925801747690635
Final loss: 0.08795824318372342
Predictions: [12.02157803  2.96081508  4.9736331 ]


In [ ]:
# Regression using dataset from a library (scikit-learn)
# Model training remains NumPy-only via LinearRegressionScratch

from sklearn.datasets import fetch_california_housing, load_diabetes

# Try California Housing first (may require download). Fallback to built-in Diabetes dataset.
try:
    data = fetch_california_housing()
    dataset_name = "California Housing"
except Exception:
    data = load_diabetes()
    dataset_name = "Diabetes (fallback, built-in)"

X_house = data.data
y_house = data.target
feature_names = data.feature_names

print("Dataset used:", dataset_name)
print("Dataset shape:", X_house.shape)
print("Target shape:", y_house.shape)
print("Features:", feature_names)

# Train/test split using NumPy (80/20)
np.random.seed(7)
idx = np.random.permutation(len(X_house))
split = int(0.8 * len(X_house))
train_idx, test_idx = idx[:split], idx[split:]

X_train, X_test = X_house[train_idx], X_house[test_idx]
y_train, y_test = y_house[train_idx], y_house[test_idx]

# Feature scaling for stable gradient descent
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0) + 1e-8
X_train_scaled = (X_train - mu) / sigma
X_test_scaled = (X_test - mu) / sigma

# Training model
house_model = LinearRegressionScratch(learning_rate=0.03, n_iters=3000)
house_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_test = house_model.predict(X_test_scaled)
mae = np.mean(np.abs(y_test - y_pred_test))
rmse = np.sqrt(np.mean((y_test - y_pred_test) ** 2))
r2 = house_model.score_r2(X_test_scaled, y_test)

print("\nTest metrics")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R^2  : {r2:.4f}")

# Predict for one sample from test set
sample_i = 0
sample_features = X_test[sample_i:sample_i+1]
sample_true = y_test[sample_i]
sample_pred = house_model.predict((sample_features - mu) / sigma)[0]

print("\nSingle sample prediction")
print("True value:", round(float(sample_true), 4))
print("Predicted value:", round(float(sample_pred), 4))

Dataset used: California Housing
Dataset shape: (20640, 8)
Target shape: (20640,)
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Test metrics
MAE  : 0.5269
RMSE : 0.7249
R^2  : 0.6056

Single sample prediction
True value: 0.969
Predicted value: 1.5675
